# DocLib Metis — Gemma 4 E4B QLoRA trên Google Colab

Notebook này dùng **chính xác cùng pipeline** với `src/training/gemma4_finetuning.py` trong dự án DocLib. Mỗi lần module dự án thay đổi, chạy `python3 scripts/build_finetune_notebook.py` để tạo lại notebook; lệnh `--check` xác minh hai bản không lệch nhau.

Pipeline thực hiện: tải `Glint-Research/Fable-5-traces` (`pi_agent`), loại bỏ `reasoning_content`, tạo mẫu prompt/completion, QLoRA NF4 cho `google/gemma-4-E4B-it`, lưu adapter, merge tùy chọn, đẩy lên Hugging Face private repo/Google Drive, chuyển F16 GGUF rồi quantize Q4_K_M và tạo `Modelfile` cho Ollama.

> Yêu cầu: bật GPU trong **Runtime → Change runtime type → GPU**. T4/L4 phù hợp cho QLoRA với batch 1; bước merge cần High-RAM hoặc A100/L4 đủ bộ nhớ. Bạn phải chấp nhận điều khoản Gemma trên Hugging Face. Dataset Fable-5-traces dùng giấy phép AGPL-3.0; hãy giữ thông tin nguồn và kiểm tra nghĩa vụ giấy phép trước khi phân phối model.


## 1. Cài môi trường Colab

In [ ]:
# Gemma 4 yêu cầu Transformers 5.5+; -U là bắt buộc trên Colab.
%pip install -q -U "transformers>=5.5,<6" "trl>=0.29" "peft>=0.18" datasets teich accelerate bitsandbytes safetensors huggingface_hub sentencepiece protobuf

# Sau lần cài đầu, nếu Colab báo phải restart runtime: Runtime → Restart session,
# rồi chạy lại notebook từ đầu.


## 2. Token Hugging Face, Drive và cấu hình

In [ ]:
import gc
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch
import transformers
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or os.getenv("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "Hãy thêm secret HF_TOKEN trong biểu tượng chìa khóa của Colab. "
        "Token cần quyền đọc model Gemma đã chấp nhận điều khoản."
    )
login(token=HF_TOKEN, add_to_git_credential=False)

MODEL_ID = "google/gemma-4-E4B-it"
DATASET_ID = "Glint-Research/Fable-5-traces"
DATASET_CONFIG = "pi_agent"
OUTPUT_ROOT = Path("/content/doclib-metis")
ADAPTER_DIR = OUTPUT_ROOT / "adapter"
MERGED_DIR = OUTPUT_ROOT / "merged"
GGUF_DIR = OUTPUT_ROOT / "gguf"

# 500 source traces là cấu hình khởi đầu an toàn. Đặt None để dùng toàn bộ dataset.
MAX_SOURCE_ROWS = 500
EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
LORA_RANK = 16
LORA_ALPHA = 32
MAX_LENGTH = 2048
SEED = 42

# Merge/GGUF cần nhiều RAM và dung lượng hơn. Adapter vẫn luôn được lưu dù hai cờ này tắt.
MERGE_MODEL = False
EXPORT_GGUF = False
SAVE_TO_DRIVE = False
PUSH_ADAPTER_TO_HUB = False
PUSH_MERGED_TO_HUB = False
HF_ADAPTER_REPO = "TEN_HF_CUA_BAN/doclib-metis-gemma4-e4b-lora"
HF_MERGED_REPO = "TEN_HF_CUA_BAN/doclib-metis-gemma4-e4b-merged"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")


## 3. Pipeline dùng chung 100% với dự án

In [ ]:
from pathlib import Path
MODULE_SOURCE = '"""Portable Gemma 4 QLoRA pipeline shared by DocLib and the Colab notebook."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport urllib.parse\nimport urllib.request\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\nDOCLIB_MODEL_NAME = "DocLib Metis"\nDEFAULT_BASE_MODEL = "google/gemma-4-E4B-it"\nDEFAULT_DATASET = "Glint-Research/Fable-5-traces"\nDEFAULT_DATASET_CONFIG = "pi_agent"\nDEFAULT_SYSTEM_PROMPT = (\n    "Bạn là DocLib Metis, trợ lý AI đa phương thức của DocLib. "\n    "Hãy trả lời đúng ngôn ngữ của người dùng, sử dụng công cụ có cấu trúc khi cần, "\n    "không tiết lộ suy luận nội bộ và không khẳng định công việc chưa thực hiện là đã hoàn tất."\n)\n\n\ndef source_sha256() -> str:\n    """Return the hash used to prove notebook/project pipeline parity."""\n    return hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n\n\ndef is_gemma4_model(model_id: str) -> bool:\n    normalized = str(model_id or "").lower().replace("_", "-")\n    return "gemma-4" in normalized or "gemma4" in normalized\n\n\ndef _text(value: Any) -> str:\n    if isinstance(value, str):\n        return value.strip()\n    if isinstance(value, list):\n        parts = []\n        for item in value:\n            if isinstance(item, str):\n                parts.append(item)\n            elif isinstance(item, dict) and item.get("type") == "text":\n                parts.append(str(item.get("text", "")))\n        return "\\n".join(part for part in parts if part).strip()\n    return ""\n\n\ndef _tool_call_text(tool_calls: Any) -> str:\n    if not tool_calls:\n        return ""\n    return "<tool_call>\\n" + json.dumps(\n        tool_calls,\n        ensure_ascii=False,\n        sort_keys=True,\n    ) + "\\n</tool_call>"\n\n\ndef normalize_trace_message(message: dict[str, Any]) -> dict[str, str] | None:\n    """Normalize a Fable trace message without importing hidden reasoning content."""\n    role = str(message.get("role", "")).lower()\n    content = _text(message.get("content"))\n    if role == "assistant":\n        tool_text = _tool_call_text(message.get("tool_calls"))\n        content = "\\n".join(part for part in (content, tool_text) if part).strip()\n    elif role == "tool":\n        tool_name = message.get("name") or message.get("tool_call_id") or "công cụ"\n        content = f"[Kết quả {tool_name}]\\n{content}".strip()\n        role = "user"\n    if role not in {"system", "user", "assistant"} or not content:\n        return None\n    return {"role": role, "content": content}\n\n\ndef fable_row_to_examples(\n    row: dict[str, Any],\n    *,\n    system_prompt: str = DEFAULT_SYSTEM_PROMPT,\n    max_history_messages: int = 24,\n) -> list[dict[str, list[dict[str, str]]]]:\n    """Expand one agent trace into assistant-target prompt/completion examples."""\n    history: list[dict[str, str]] = []\n    examples = []\n    for raw_message in row.get("messages") or []:\n        message = normalize_trace_message(raw_message)\n        if message is None:\n            continue\n        if message["role"] == "assistant":\n            prompt = history[-max_history_messages:]\n            if not prompt or prompt[0]["role"] != "system":\n                prompt = [{"role": "system", "content": system_prompt}, *prompt]\n            if any(item["role"] == "user" for item in prompt):\n                examples.append({"prompt": prompt, "completion": [message]})\n        history.append(message)\n    return examples\n\n\ndef load_fable_training_dataset(\n    *,\n    dataset_id: str = DEFAULT_DATASET,\n    dataset_config: str = DEFAULT_DATASET_CONFIG,\n    split: str = "train",\n    max_rows: int | None = None,\n    seed: int = 42,\n    max_history_messages: int = 24,\n):\n    """Load and normalize the official Fable trace dataset for text SFT.\n\n    The repository currently contains heterogeneous raw JSONL shards. The Hub\n    datasets server exposes the repository\'s correctly normalized ``pi_agent``\n    view, so it is used only when ``load_dataset`` raises while decoding a raw\n    shard.\n    """\n    from datasets import Dataset, load_dataset\n\n    def normalized_examples(rows):\n        normalized = []\n        for row in rows:\n            normalized.extend(\n                fable_row_to_examples(\n                    row,\n                    max_history_messages=max_history_messages,\n                )\n            )\n        return normalized\n\n    try:\n        if max_rows:\n            source = load_dataset(\n                dataset_id,\n                dataset_config,\n                split=split,\n                streaming=True,\n            ).take(max_rows)\n        else:\n            source = load_dataset(dataset_id, dataset_config, split=split)\n        examples = normalized_examples(source)\n    except Exception as load_error:\n        rows = []\n        offset = 0\n        target = int(max_rows) if max_rows else None\n        while target is None or len(rows) < target:\n            length = min(100, (target - len(rows)) if target else 100)\n            query = urllib.parse.urlencode(\n                {\n                    "dataset": dataset_id,\n                    "config": dataset_config,\n                    "split": split,\n                    "offset": offset,\n                    "length": length,\n                }\n            )\n            request = urllib.request.Request(\n                f"https://datasets-server.huggingface.co/rows?{query}"\n            )\n            hf_token = os.getenv("HF_TOKEN", "").strip()\n            if hf_token:\n                request.add_header("Authorization", f"Bearer {hf_token}")\n            try:\n                with urllib.request.urlopen(request, timeout=60) as response:\n                    payload = json.load(response)\n            except Exception as fallback_error:\n                raise RuntimeError("fable_dataset_loading_failed") from ExceptionGroup(\n                    "load_dataset_and_dataset_server_failed",\n                    [load_error, fallback_error],\n                )\n            page = [item.get("row", {}) for item in payload.get("rows", [])]\n            rows.extend(page)\n            offset += len(page)\n            total = int(payload.get("num_rows_total", offset) or offset)\n            if not page or offset >= total:\n                break\n        examples = normalized_examples(rows[:target] if target else rows)\n    if not examples:\n        raise ValueError("fable_dataset_has_no_trainable_assistant_messages")\n    return Dataset.from_list(examples).shuffle(seed=seed)\n\n\ndef doclib_samples_to_dataset(\n    samples: Iterable[dict[str, Any]],\n    *,\n    system_prompt: str = DEFAULT_SYSTEM_PROMPT,\n):\n    """Convert DocLib\'s instruction/input/output records to the same SFT schema."""\n    from datasets import Dataset\n\n    rows = []\n    for sample in samples:\n        instruction = _text(sample.get("instruction"))\n        context = _text(sample.get("input"))\n        output = _text(sample.get("output"))\n        prompt = "\\n".join(part for part in (instruction, context) if part).strip()\n        if prompt and output:\n            rows.append(\n                {\n                    "prompt": [\n                        {"role": "system", "content": system_prompt},\n                        {"role": "user", "content": prompt},\n                    ],\n                    "completion": [{"role": "assistant", "content": output}],\n                }\n            )\n    if not rows:\n        raise ValueError("finetuning_dataset_has_no_valid_samples")\n    return Dataset.from_list(rows)\n\n\ndef validate_colab_runtime() -> dict[str, Any]:\n    """Fail early instead of pretending that an 8B-weight Gemma model can train on CPU."""\n    import torch\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("gemma4_qlora_requires_cuda_gpu")\n    major, _minor = torch.cuda.get_device_capability()\n    return {\n        "device": torch.cuda.get_device_name(0),\n        "bf16": bool(torch.cuda.is_bf16_supported()),\n        "compute_capability": major,\n        "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),\n    }\n\n\ndef load_gemma4_qlora(\n    model_id: str = DEFAULT_BASE_MODEL,\n    *,\n    hf_token: str | None = None,\n    revision: str | None = None,\n):\n    """Load Gemma 4 with the official multimodal auto class and NF4 QLoRA."""\n    import torch\n    from peft import prepare_model_for_kbit_training\n    from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig\n\n    runtime = validate_colab_runtime()\n    compute_dtype = torch.bfloat16 if runtime["bf16"] else torch.float16\n    quantization = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_compute_dtype=compute_dtype,\n        bnb_4bit_use_double_quant=True,\n    )\n    common = {"token": hf_token}\n    if revision:\n        common["revision"] = revision\n    processor = AutoProcessor.from_pretrained(model_id, **common)\n    model = AutoModelForMultimodalLM.from_pretrained(\n        model_id,\n        quantization_config=quantization,\n        device_map={"": torch.cuda.current_device()},\n        dtype=compute_dtype,\n        low_cpu_mem_usage=True,\n        **common,\n    )\n    model = prepare_model_for_kbit_training(\n        model,\n        use_gradient_checkpointing=True,\n    )\n    model.gradient_checkpointing_enable(\n        gradient_checkpointing_kwargs={"use_reentrant": False}\n    )\n    model.config.use_cache = False\n    return model, processor, runtime\n\n\ndef create_lora_config(rank: int = 16, alpha: int = 32, dropout: float = 0.05):\n    """Create the language-backbone adapter shared by local and Colab training."""\n    from peft import LoraConfig\n\n    return LoraConfig(\n        r=int(rank),\n        lora_alpha=int(alpha),\n        lora_dropout=float(dropout),\n        target_modules=[\n            "q_proj",\n            "k_proj",\n            "v_proj",\n            "o_proj",\n            "gate_proj",\n            "up_proj",\n            "down_proj",\n        ],\n        bias="none",\n        task_type="CAUSAL_LM",\n    )\n\n\ndef train_gemma4_qlora(\n    *,\n    dataset,\n    output_dir: str | Path,\n    model_id: str = DEFAULT_BASE_MODEL,\n    hf_token: str | None = None,\n    revision: str | None = None,\n    epochs: int = 1,\n    batch_size: int = 1,\n    gradient_accumulation_steps: int = 8,\n    learning_rate: float = 2e-4,\n    lora_rank: int = 16,\n    lora_alpha: int = 32,\n    max_length: int = 2048,\n    update_callback: Callable[[dict[str, Any]], None] | None = None,\n):\n    """Train a text/tool-use QLoRA adapter while retaining Gemma 4 multimodality."""\n    import torch\n    from peft import get_peft_model\n    from transformers import TrainerCallback\n    from trl import SFTConfig, SFTTrainer\n\n    callback = update_callback or (lambda _data: None)\n    callback({"progress": 10, "status": "running"})\n    model, processor, runtime = load_gemma4_qlora(\n        model_id,\n        hf_token=hf_token,\n        revision=revision,\n    )\n    model = get_peft_model(\n        model,\n        create_lora_config(lora_rank, lora_alpha),\n    )\n    callback({"progress": 20})\n\n    total_steps = max(\n        1,\n        math.ceil(len(dataset) / max(1, batch_size * gradient_accumulation_steps))\n        * int(epochs),\n    )\n\n    class ProgressCallback(TrainerCallback):\n        def on_log(self, args, state, control, logs=None, **kwargs):\n            logs = logs or {}\n            progress = 25 + (state.global_step / total_steps) * 65\n            callback(\n                {\n                    "progress": round(min(progress, 90), 1),\n                    "current_loss": round(float(logs.get("loss", 0) or 0), 6),\n                    "current_epoch": min(\n                        int(epochs),\n                        max(1, math.ceil(float(logs.get("epoch", 0) or 0))),\n                    ),\n                }\n            )\n\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    bf16 = bool(runtime["bf16"])\n    args = SFTConfig(\n        output_dir=str(output_dir),\n        num_train_epochs=int(epochs),\n        per_device_train_batch_size=int(batch_size),\n        gradient_accumulation_steps=int(gradient_accumulation_steps),\n        learning_rate=float(learning_rate),\n        logging_steps=1,\n        save_strategy="epoch",\n        save_total_limit=2,\n        bf16=bf16,\n        fp16=not bf16,\n        optim="paged_adamw_8bit",\n        max_length=int(max_length),\n        packing=False,\n        completion_only_loss=True,\n        gradient_checkpointing=True,\n        gradient_checkpointing_kwargs={"use_reentrant": False},\n        report_to="none",\n        remove_unused_columns=False,\n    )\n    trainer = SFTTrainer(\n        model=model,\n        processing_class=processor,\n        train_dataset=dataset,\n        args=args,\n        callbacks=[ProgressCallback()],\n    )\n    result = trainer.train()\n    trainer.model.save_pretrained(output_dir, safe_serialization=True)\n    processor.save_pretrained(output_dir)\n    final_loss = float(result.metrics.get("train_loss", 0) or 0)\n    callback({"progress": 92, "current_loss": round(final_loss, 6)})\n    return {\n        "trainer": trainer,\n        "processor": processor,\n        "adapter_path": str(output_dir),\n        "final_loss": final_loss,\n        "runtime": runtime,\n    }\n\n\ndef merge_gemma4_adapter(\n    *,\n    adapter_path: str | Path,\n    merged_path: str | Path,\n    model_id: str = DEFAULT_BASE_MODEL,\n    hf_token: str | None = None,\n    revision: str | None = None,\n):\n    """Merge the QLoRA adapter into an unquantized multimodal Gemma 4 model."""\n    import torch\n    from peft import PeftModel\n    from transformers import AutoModelForMultimodalLM, AutoProcessor\n\n    common = {"token": hf_token}\n    if revision:\n        common["revision"] = revision\n    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float16\n    base = AutoModelForMultimodalLM.from_pretrained(\n        model_id,\n        device_map="cpu",\n        dtype=dtype,\n        low_cpu_mem_usage=True,\n        **common,\n    )\n    merged = PeftModel.from_pretrained(base, str(adapter_path)).merge_and_unload(\n        safe_merge=True\n    )\n    processor = AutoProcessor.from_pretrained(model_id, **common)\n    merged_path = Path(merged_path)\n    merged_path.mkdir(parents=True, exist_ok=True)\n    merged.save_pretrained(\n        merged_path,\n        safe_serialization=True,\n        max_shard_size="4GB",\n    )\n    processor.save_pretrained(merged_path)\n    return str(merged_path)\n\n\ndef _find_quantizer(llama_cpp_dir: Path) -> Path | None:\n    candidates = [\n        llama_cpp_dir / "build/bin/llama-quantize",\n        llama_cpp_dir / "llama-quantize",\n        Path(shutil.which("llama-quantize") or ""),\n    ]\n    return next((path for path in candidates if path and path.is_file()), None)\n\n\ndef write_ollama_modelfile(gguf_path: str | Path, output_path: str | Path) -> str:\n    """Write a portable Ollama definition for the merged DocLib Metis model."""\n    gguf_name = Path(gguf_path).name\n    content = (\n        f"FROM ./{gguf_name}\\n"\n        "PARAMETER temperature 1.0\\n"\n        "PARAMETER top_p 0.95\\n"\n        "PARAMETER top_k 64\\n"\n        "PARAMETER num_ctx 4096\\n"\n        f\'SYSTEM """{DEFAULT_SYSTEM_PROMPT}"""\\n\'\n    )\n    output_path = Path(output_path)\n    output_path.write_text(content, encoding="utf-8")\n    return str(output_path)\n\n\ndef export_gguf_artifacts(\n    *,\n    merged_path: str | Path,\n    output_dir: str | Path,\n    llama_cpp_dir: str | Path,\n    quantization: str = "Q4_K_M",\n    timeout_seconds: int = 7200,\n) -> dict[str, str]:\n    """Convert HF weights to F16 GGUF, then quantize with llama-quantize."""\n    merged_path = Path(merged_path)\n    output_dir = Path(output_dir)\n    llama_cpp_dir = Path(llama_cpp_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    converter = llama_cpp_dir / "convert_hf_to_gguf.py"\n    if not converter.is_file():\n        raise FileNotFoundError("llama_cpp_converter_not_found")\n    f16_path = output_dir / "doclib-metis-F16.gguf"\n    subprocess.run(\n        [\n            os.sys.executable,\n            str(converter),\n            str(merged_path),\n            "--outfile",\n            str(f16_path),\n            "--outtype",\n            "f16",\n        ],\n        check=True,\n        timeout=timeout_seconds,\n    )\n    result = {"f16_gguf_path": str(f16_path)}\n    quantizer = _find_quantizer(llama_cpp_dir)\n    if quantizer:\n        quantized_path = output_dir / f"doclib-metis-{quantization}.gguf"\n        subprocess.run(\n            [str(quantizer), str(f16_path), str(quantized_path), quantization],\n            check=True,\n            timeout=timeout_seconds,\n        )\n        result["gguf_path"] = str(quantized_path)\n    else:\n        result["gguf_path"] = str(f16_path)\n    result["modelfile_path"] = write_ollama_modelfile(\n        result["gguf_path"],\n        output_dir / "Modelfile",\n    )\n    return result\n\n\ndef build_artifact_manifest(\n    *,\n    adapter_path: str | Path,\n    merged_path: str | Path | None = None,\n    gguf_path: str | Path | None = None,\n) -> dict[str, Any]:\n    """Describe portable outputs for Hub, Drive and local DocLib deployment."""\n    return {\n        "model_name": DOCLIB_MODEL_NAME,\n        "base_model": DEFAULT_BASE_MODEL,\n        "dataset": DEFAULT_DATASET,\n        "adapter_path": str(adapter_path),\n        "merged_path": str(merged_path) if merged_path else None,\n        "gguf_path": str(gguf_path) if gguf_path else None,\n        "pipeline_sha256": source_sha256(),\n    }\n'
PORTABLE_MODULE = Path('/content/gemma4_finetuning.py')
PORTABLE_MODULE.write_text(MODULE_SOURCE, encoding='utf-8')
print(f'Đã đồng bộ pipeline dùng chung: {PORTABLE_MODULE}')


In [ ]:
sys.path.insert(0, "/content")
from gemma4_finetuning import (
    DOCLIB_MODEL_NAME,
    DEFAULT_BASE_MODEL,
    DEFAULT_DATASET,
    build_artifact_manifest,
    export_gguf_artifacts,
    load_fable_training_dataset,
    merge_gemma4_adapter,
    source_sha256,
    train_gemma4_qlora,
    validate_colab_runtime,
)

assert MODEL_ID == DEFAULT_BASE_MODEL
assert DATASET_ID == DEFAULT_DATASET
runtime = validate_colab_runtime()
print(DOCLIB_MODEL_NAME, runtime)
print("SHA-256 pipeline:", source_sha256())


## 4. Tải và chuẩn hóa Fable-5-traces

In [ ]:
dataset = load_fable_training_dataset(
    dataset_id=DATASET_ID,
    dataset_config=DATASET_CONFIG,
    max_rows=MAX_SOURCE_ROWS,
    seed=SEED,
    max_history_messages=24,
)
split = dataset.train_test_split(test_size=min(0.02, max(1 / len(dataset), 0.005)), seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Mẫu train: {len(train_dataset):,}; mẫu kiểm tra: {len(eval_dataset):,}")
print(json.dumps(train_dataset[0], ensure_ascii=False, indent=2)[:4000])
assert "reasoning_content" not in json.dumps(train_dataset[0], ensure_ascii=False)


## 5. QLoRA DocLib Metis

In [ ]:
def show_progress(data):
    print(
        f"Tiến độ {data.get('progress', 0):>5}% | "
        f"epoch={data.get('current_epoch', '-')} | "
        f"loss={data.get('current_loss', '-')}"
    )

training = train_gemma4_qlora(
    dataset=train_dataset,
    output_dir=ADAPTER_DIR,
    model_id=MODEL_ID,
    hf_token=HF_TOKEN,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    max_length=MAX_LENGTH,
    update_callback=show_progress,
)
trainer = training["trainer"]
processor = training["processor"]
print("Adapter:", training["adapter_path"])
print("Train loss:", training["final_loss"])


## 6. Kiểm tra adapter ngay trên Colab

In [ ]:
messages = [
    {"role": "system", "content": "Bạn là DocLib Metis."},
    {"role": "user", "content": "Giới thiệu ngắn gọn bạn là ai."},
]
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
).to(trainer.model.device)
input_length = inputs["input_ids"].shape[-1]
with torch.inference_mode():
    generated = trainer.model.generate(**inputs, max_new_tokens=96)
response = processor.decode(generated[0][input_length:], skip_special_tokens=False)
print(processor.parse_response(response, prefix=inputs["input_ids"]))


## 7. Đẩy adapter lên Hugging Face private repo

In [ ]:
api = HfApi(token=HF_TOKEN)
if PUSH_ADAPTER_TO_HUB:
    if HF_ADAPTER_REPO.startswith("TEN_HF_CUA_BAN/"):
        raise ValueError("Hãy thay TEN_HF_CUA_BAN trong HF_ADAPTER_REPO")
    api.create_repo(HF_ADAPTER_REPO, private=True, exist_ok=True)
    api.upload_folder(
        repo_id=HF_ADAPTER_REPO,
        folder_path=str(ADAPTER_DIR),
        commit_message="DocLib Metis Gemma 4 E4B QLoRA adapter",
    )
    print("Đã đẩy adapter private:", HF_ADAPTER_REPO)
else:
    print("Adapter đã lưu tại", ADAPTER_DIR, "— bật PUSH_ADAPTER_TO_HUB khi sẵn sàng.")


## 8. Merge adapter vào Gemma 4 (khuyên dùng trước khi xuất GGUF)

In [ ]:
merged_path = None
if MERGE_MODEL or EXPORT_GGUF or PUSH_MERGED_TO_HUB:
    # Giải phóng QLoRA khỏi VRAM trước khi tải base model không quantize.
    del trainer
    del training
    gc.collect()
    torch.cuda.empty_cache()
    merged_path = merge_gemma4_adapter(
        adapter_path=ADAPTER_DIR,
        merged_path=MERGED_DIR,
        model_id=MODEL_ID,
        hf_token=HF_TOKEN,
    )
    print("Merged model:", merged_path)
else:
    print("Đang giữ adapter riêng. Bật MERGE_MODEL nếu Colab có High-RAM.")


## 9. Đẩy merged model lên Hugging Face private repo

In [ ]:
if PUSH_MERGED_TO_HUB:
    if not merged_path:
        raise RuntimeError("Bật MERGE_MODEL trước khi push merged model")
    if HF_MERGED_REPO.startswith("TEN_HF_CUA_BAN/"):
        raise ValueError("Hãy thay TEN_HF_CUA_BAN trong HF_MERGED_REPO")
    api.create_repo(HF_MERGED_REPO, private=True, exist_ok=True)
    api.upload_folder(
        repo_id=HF_MERGED_REPO,
        folder_path=str(MERGED_DIR),
        commit_message="DocLib Metis merged Gemma 4 E4B",
    )
    print("Đã đẩy merged model private:", HF_MERGED_REPO)


## 10. Xuất GGUF F16 → Q4_K_M và Modelfile Ollama

In [ ]:
gguf_artifacts = {}
if EXPORT_GGUF:
    if not merged_path:
        raise RuntimeError("GGUF yêu cầu merged model; bật MERGE_MODEL hoặc EXPORT_GGUF từ đầu")
    llama_cpp = Path("/content/llama.cpp")
    if not llama_cpp.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(llama_cpp)],
            check=True,
        )
    subprocess.run(
        [
            "cmake", "-S", str(llama_cpp), "-B", str(llama_cpp / "build"),
            "-DGGML_NATIVE=OFF", "-DLLAMA_CURL=OFF",
        ],
        check=True,
    )
    subprocess.run(
        ["cmake", "--build", str(llama_cpp / "build"), "--target", "llama-quantize", "-j2"],
        check=True,
    )
    gguf_artifacts = export_gguf_artifacts(
        merged_path=MERGED_DIR,
        output_dir=GGUF_DIR,
        llama_cpp_dir=llama_cpp,
        quantization="Q4_K_M",
    )
    print(json.dumps(gguf_artifacts, ensure_ascii=False, indent=2))
else:
    print("Bật EXPORT_GGUF để tạo F16, Q4_K_M và Modelfile.")


## 11. Lưu Google Drive và manifest

In [ ]:
manifest = build_artifact_manifest(
    adapter_path=ADAPTER_DIR,
    merged_path=merged_path,
    gguf_path=gguf_artifacts.get("gguf_path"),
)
(OUTPUT_ROOT / "artifact-manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_target = Path("/content/drive/MyDrive/DocLib-Metis")
    if drive_target.exists():
        shutil.rmtree(drive_target)
    shutil.copytree(OUTPUT_ROOT, drive_target)
    print("Đã lưu vào", drive_target)

print(json.dumps(manifest, ensure_ascii=False, indent=2))


## 12. Áp vào DocLib local

Sau khi tải thư mục `gguf/` về máy, chạy tại thư mục đó:

```bash
ollama create doclib-metis -f Modelfile
ollama run doclib-metis "Xin chào, bạn là ai?"
```

Sau khi test thành công, đổi `.env` của dự án:

```dotenv
LLM_MODEL=doclib-metis:latest
```

rồi tái tạo riêng các dịch vụ gọi model:

```bash
docker compose up -d --force-recreate agentic_ai rag usage
```

Nếu chỉ tải adapter, giữ đúng base model `google/gemma-4-E4B-it` khi merge. Không gắn adapter vào một bản Gemma khác hoặc một quantization khác. Với llama.cpp multimodal, projector cần được xuất/test riêng; GGUF chính trong notebook là đường triển khai text/tool-use an toàn nhất cho dataset Fable vốn chỉ có text traces. Hãy giữ model Ollama cũ cho audio/image cho đến khi bản GGUF mới vượt qua kiểm thử đa phương thức.
